# Reddit Resume Scraper for r/csMajors (No Authentication Required)

This notebook scrapes student resumes from the r/csMajors subreddit using Reddit's JSON API (no credentials needed!), including:
- Direct file attachments on Reddit posts
- Imgur links embedded in posts
- Imgur albums
- Reddit galleries

The scraper will save all resumes with metadata about the post.

In [1]:
import requests
import re
import os
import json
import time
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse, urljoin
import pandas as pd
from tqdm import tqdm

print("✓ All packages imported successfully!")
print("Note: No Reddit API credentials needed for this scraper!")

✓ All packages imported successfully!
Note: No Reddit API credentials needed for this scraper!


In [2]:
# Setup requests session with proper headers
# Reddit's JSON API is public and doesn't require authentication!

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

print("✓ HTTP session configured!")
print("✓ Ready to scrape Reddit (no credentials needed)")

✓ HTTP session configured!
✓ Ready to scrape Reddit (no credentials needed)


In [3]:
# Create directories for storing resumes
output_dir = Path("resumes")
output_dir.mkdir(exist_ok=True)

# Directory structure:
# resumes/
#   ├── files/          # All downloaded resume files
#   └── metadata.json   # Metadata about each resume

files_dir = output_dir / "files"
files_dir.mkdir(exist_ok=True)

print(f"Output directory: {output_dir.absolute()}")
print(f"Files directory: {files_dir.absolute()}")

Output directory: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes
Files directory: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes/files


In [4]:
def extract_imgur_urls(text):
    """Extract imgur URLs from text (single images and albums)"""
    if not text:
        return []
    
    imgur_patterns = [
        r'https?://(?:i\.)?imgur\.com/([a-zA-Z0-9]+)\.?(?:png|jpg|jpeg|gif|pdf)?',
        r'https?://imgur\.com/(?:a/|gallery/)([a-zA-Z0-9]+)',
    ]
    
    urls = []
    for pattern in imgur_patterns:
        matches = re.findall(pattern, text)
        urls.extend(matches)
    
    return list(set(urls))  # Remove duplicates

def get_imgur_direct_link(imgur_id):
    """Convert imgur ID to direct download link"""
    # Try common image extensions
    extensions = ['jpg', 'png', 'jpeg', 'pdf']
    for ext in extensions:
        url = f"https://i.imgur.com/{imgur_id}.{ext}"
        try:
            response = session.head(url, timeout=5)
            if response.status_code == 200:
                return url
        except:
            continue
    return None

def download_file(url, output_path, timeout=30):
    """Download a file from URL to output_path"""
    try:
        response = session.get(url, timeout=timeout, stream=True)
        response.raise_for_status()
        
        with open(output_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        return True
    except Exception as e:
        print(f"Error downloading {url}: {str(e)}")
        return False

def get_file_extension_from_url(url):
    """Extract file extension from URL"""
    path = urlparse(url).path
    ext = os.path.splitext(path)[1]
    if ext:
        return ext.lower()
    return '.jpg'  # Default to .jpg for imgur

def fetch_reddit_json(url, params=None):
    """Fetch JSON data from Reddit's API"""
    try:
        # Add .json to the URL if not present
        if not url.endswith('.json'):
            url = url.rstrip('/') + '.json'
        
        response = session.get(url, params=params, timeout=10)
        response.raise_for_status()
        time.sleep(2)  # Be respectful to Reddit's servers
        return response.json()
    except Exception as e:
        print(f"Error fetching {url}: {str(e)}")
        return None

print("✓ Helper functions loaded successfully!")

✓ Helper functions loaded successfully!


In [5]:
def process_reddit_post(post_json):
    """Process a Reddit post from JSON data and extract resume files"""
    data = post_json['data']
    
    post_data = {
        'post_id': data.get('id', ''),
        'title': data.get('title', ''),
        'author': data.get('author', '[deleted]'),
        'created_utc': data.get('created_utc', 0),
        'created_date': datetime.fromtimestamp(data.get('created_utc', 0)).strftime('%Y-%m-%d %H:%M:%S'),
        'url': data.get('url', ''),
        'selftext': data.get('selftext', ''),
        'score': data.get('score', 0),
        'num_comments': data.get('num_comments', 0),
        'permalink': f"https://reddit.com{data.get('permalink', '')}",
        'downloaded_files': []
    }
    
    downloaded_count = 0
    post_id = post_data['post_id']
    
    # Check if the post itself is an image/file
    url = data.get('url', '')
    if url:
        # Direct image/PDF links
        if any(url.endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.pdf', '.gif']):
            ext = get_file_extension_from_url(url)
            filename = f"{post_id}_direct{ext}"
            output_path = files_dir / filename
            
            if download_file(url, output_path):
                post_data['downloaded_files'].append({
                    'filename': filename,
                    'source': 'direct_post',
                    'url': url
                })
                downloaded_count += 1
        
        # Reddit's own image hosting (i.redd.it)
        elif 'i.redd.it' in url:
            ext = get_file_extension_from_url(url)
            filename = f"{post_id}_reddit{ext}"
            output_path = files_dir / filename
            
            if download_file(url, output_path):
                post_data['downloaded_files'].append({
                    'filename': filename,
                    'source': 'reddit_media',
                    'url': url
                })
                downloaded_count += 1
    
    # Check for imgur links in post text and URL
    text_to_check = f"{post_data['selftext']} {post_data['url']}"
    imgur_ids = extract_imgur_urls(text_to_check)
    
    for idx, imgur_id in enumerate(imgur_ids):
        direct_url = get_imgur_direct_link(imgur_id)
        if direct_url:
            ext = get_file_extension_from_url(direct_url)
            filename = f"{post_id}_imgur_{idx}{ext}"
            output_path = files_dir / filename
            
            if download_file(direct_url, output_path):
                post_data['downloaded_files'].append({
                    'filename': filename,
                    'source': 'imgur',
                    'url': direct_url,
                    'imgur_id': imgur_id
                })
                downloaded_count += 1
    
    # Check for Reddit gallery
    if data.get('is_gallery'):
        try:
            gallery_data = data.get('gallery_data', {})
            media_metadata = data.get('media_metadata', {})
            
            for idx, item in enumerate(gallery_data.get('items', [])):
                media_id = item.get('media_id', '')
                if media_id in media_metadata:
                    media_info = media_metadata[media_id]
                    
                    # Get the highest quality image
                    if 's' in media_info:
                        img_url = media_info['s'].get('u', '')
                        if img_url:
                            # Decode HTML entities in URL
                            img_url = img_url.replace('&amp;', '&')
                            ext = get_file_extension_from_url(img_url)
                            filename = f"{post_id}_gallery_{idx}{ext}"
                            output_path = files_dir / filename
                            
                            if download_file(img_url, output_path):
                                post_data['downloaded_files'].append({
                                    'filename': filename,
                                    'source': 'reddit_gallery',
                                    'url': img_url
                                })
                                downloaded_count += 1
        except Exception as e:
            print(f"Error processing gallery for {post_id}: {str(e)}")
    
    return post_data, downloaded_count

print("✓ Reddit post processor loaded!")

✓ Reddit post processor loaded!


In [6]:
# Configuration for scraping
SUBREDDIT_NAME = "csMajors"
SEARCH_QUERY = "resume"
LIMIT = 1000  # Maximum number of posts to scrape
SORT_BY = "relevance"  # Options: "relevance", "hot", "top", "new", "comments"
TIME_FILTER = "all"  # Options: "all", "day", "week", "month", "year"

print(f"Configuration:")
print(f"  Subreddit: r/{SUBREDDIT_NAME}")
print(f"  Search query: '{SEARCH_QUERY}'")
print(f"  Limit: {LIMIT}")
print(f"  Sort by: {SORT_BY}")
print(f"  Time filter: {TIME_FILTER}")
print(f"\nNote: Reddit's JSON API returns ~25 posts per request")
print(f"      We'll paginate through results to reach the limit")

Configuration:
  Subreddit: r/csMajors
  Search query: 'resume'
  Limit: 1000
  Sort by: relevance
  Time filter: all

Note: Reddit's JSON API returns ~25 posts per request
      We'll paginate through results to reach the limit


In [7]:
# Main scraping execution using Reddit's JSON API
print(f"Searching r/{SUBREDDIT_NAME} for '{SEARCH_QUERY}'...\n")

all_posts_data = []
total_files_downloaded = 0
posts_with_files = 0

# Build the search URL
base_url = f"https://www.reddit.com/r/{SUBREDDIT_NAME}/search.json"
params = {
    'q': SEARCH_QUERY,
    'restrict_sr': 'on',  # Restrict to this subreddit
    'sort': SORT_BY,
    't': TIME_FILTER,
    'limit': 100  # Max per request
}

after = None
posts_processed = 0

# Paginate through results
with tqdm(total=LIMIT, desc="Fetching posts") as pbar:
    while posts_processed < LIMIT:
        # Add pagination parameter
        if after:
            params['after'] = after
        
        # Fetch a batch of posts
        data = fetch_reddit_json(base_url, params)
        
        if not data or 'data' not in data:
            print("No more data available")
            break
        
        posts = data['data'].get('children', [])
        after = data['data'].get('after')  # For pagination
        
        if not posts:
            print("No more posts found")
            break
        
        # Process each post in this batch
        for post in posts:
            if posts_processed >= LIMIT:
                break
                
            try:
                post_data, file_count = process_reddit_post(post)
                all_posts_data.append(post_data)
                posts_processed += 1
                pbar.update(1)
                
                if file_count > 0:
                    posts_with_files += 1
                    total_files_downloaded += file_count
                    title_preview = post_data['title'][:60] + '...' if len(post_data['title']) > 60 else post_data['title']
                    print(f"✓ {post_data['post_id']}: '{title_preview}' - {file_count} file(s)")
                
            except Exception as e:
                print(f"✗ Error processing post: {str(e)}")
                continue
        
        # If there's no 'after' token, we've reached the end
        if not after:
            print("Reached end of search results")
            break
        
        # Be respectful - wait between batches
        time.sleep(2)

print(f"\n{'='*70}")
print(f"SCRAPING COMPLETE!")
print(f"{'='*70}")
print(f"Total posts processed: {len(all_posts_data)}")
print(f"Posts with files: {posts_with_files}")
print(f"Total files downloaded: {total_files_downloaded}")
print(f"Files saved to: {files_dir.absolute()}")
print(f"{'='*70}")

Searching r/csMajors for 'resume'...



Fetching posts:   1%|          | 11/1000 [00:03<03:41,  4.46it/s]

✓ 1nhwbhc: 'My job search' - 1 file(s)
✓ 1gisnpg: 'I am done, I am so relieved' - 1 file(s)


Fetching posts:   4%|▎         | 36/1000 [00:04<00:48, 19.92it/s]

✓ 1e8zn36: 'A State your IQ in your Resumes' - 1 file(s)
✓ 1nnsdyn: 'does ts mean I got interview or is it another resume screeni...' - 1 file(s)


...

✓ 1hmsul4: 'Finally got an offer! 😭+ my advice ' - 1 file(s)
✓ 185n0f5: 'New grad job hunt (app number is probably + or - 150)' - 1 file(s)


Fetching posts:  23%|██▎       | 231/1000 [00:21<01:06, 11.52it/s]

✓ 1ov4zxf: 'Clearly a change is needed…' - 1 file(s)
✓ 1i3hjf7: 'We are so back' - 1 file(s)


Fetching posts:  24%|██▎       | 235/1000 [00:21<01:10, 10.85it/s]

✓ 1j34prf: 'Adam, 45, SRE. Only wants remote. Never been on LinkedIn.' - 1 file(s)
Reached end of search results

SCRAPING COMPLETE!
Total posts processed: 235
Posts with files: 31
Total files downloaded: 34
Files saved to: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes/files


In [8]:
# Save metadata to JSON file
metadata_file = output_dir / "metadata.json"

with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(all_posts_data, f, indent=2, ensure_ascii=False)

print(f"Metadata saved to: {metadata_file.absolute()}")

# Also create a CSV for easier analysis
posts_df = pd.DataFrame([
    {
        'post_id': post['post_id'],
        'title': post['title'],
        'author': post['author'],
        'created_date': post['created_date'],
        'score': post['score'],
        'num_comments': post['num_comments'],
        'num_files': len(post['downloaded_files']),
        'permalink': post['permalink']
    }
    for post in all_posts_data
])

csv_file = output_dir / "posts_summary.csv"
posts_df.to_csv(csv_file, index=False)
print(f"Posts summary saved to: {csv_file.absolute()}")

Metadata saved to: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes/metadata.json
Posts summary saved to: /Users/anirudhannabathula/Desktop/ds3/25-26_Projects/TalentLens_Public/data/reddit/resumes/posts_summary.csv


In [9]:
# Display summary statistics
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

# Posts with files
posts_with_files_df = posts_df[posts_df['num_files'] > 0]

print(f"\nTotal posts found: {len(posts_df)}")
print(f"Posts with resume files: {len(posts_with_files_df)}")
print(f"Total resume files downloaded: {posts_df['num_files'].sum()}")

if len(posts_with_files_df) > 0:
    print(f"\nTop posts by engagement:")
    print(posts_with_files_df.nlargest(10, 'score')[['title', 'author', 'score', 'num_files', 'created_date']])
    
    print(f"\n\nFile type distribution:")
    file_extensions = []
    for post in all_posts_data:
        for file_info in post['downloaded_files']:
            ext = os.path.splitext(file_info['filename'])[1]
            file_extensions.append(ext)
    
    ext_df = pd.DataFrame({'extension': file_extensions})
    print(ext_df['extension'].value_counts())
    
    print(f"\n\nSource distribution:")
    sources = []
    for post in all_posts_data:
        for file_info in post['downloaded_files']:
            sources.append(file_info['source'])
    
    source_df = pd.DataFrame({'source': sources})
    print(source_df['source'].value_counts())
else:
    print("\nNo resume files were found in the scraped posts.")

SUMMARY STATISTICS

Total posts found: 235
Posts with resume files: 31
Total resume files downloaded: 34

Top posts by engagement:
                                                 title                author  \
10                         I am done, I am so relieved          Random_Knowl   
58      Years of hard work finally starting to pay off  Odd-Personality-1294   
96                                      Bro Is Cooking             Kalex8876   
115       lazy guide from unemployed to employed in CS        theheadstarter   
5                                        My job search          fin_Meen_kan   
85                          we are just doing it wrong    roguethrowaway0999   
234  Adam, 45, SRE. Only wants remote. Never been o...       Lazy-Store-2971   
105                                  Bruh aint no wayy       Lazy-Store-2971   
162  Crying @ this dude coming into our sub for fre...          Condomphobic   
213           Applied to 60 apple jobs today alone lol     Current-Pa

In [ ]:
# Display sample posts with downloaded resumes
print("=" * 60)
print("SAMPLE POSTS WITH RESUMES")
print("=" * 60)

posts_with_downloads = [post for post in all_posts_data if len(post['downloaded_files']) > 0]

if posts_with_downloads:
    for i, post in enumerate(posts_with_downloads[:5]):  # Show first 5
        print(f"\n{i+1}. Post ID: {post['post_id']}")
        print(f"   Title: {post['title']}")
        print(f"   Author: {post['author']}")
        print(f"   Date: {post['created_date']}")
        print(f"   Score: {post['score']}")
        print(f"   Link: {post['permalink']}")
        print(f"   Files downloaded ({len(post['downloaded_files'])}):")
        for file_info in post['downloaded_files']:
            print(f"      - {file_info['filename']} (from {file_info['source']})")
else:
    print("\nNo posts with downloaded files to display.")

SAMPLE POSTS WITH RESUMES

1. Post ID: 1nhwbhc
   Title: My job search
   Author: fin_Meen_kan
   Date: 2025-09-15 12:57:18
   Score: 4394
   Link: https://reddit.com/r/csMajors/comments/1nhwbhc/my_job_search/
   Files downloaded (1):
      - 1nhwbhc_direct.png (from direct_post)

2. Post ID: 1gisnpg
   Title: I am done, I am so relieved
   Author: Random_Knowl
   Date: 2024-11-03 09:48:45
   Score: 12709
   Link: https://reddit.com/r/csMajors/comments/1gisnpg/i_am_done_i_am_so_relieved/
   Files downloaded (1):
      - 1gisnpg_direct.png (from direct_post)

3. Post ID: 1e8zn36
   Title: A State your IQ in your Resumes
   Author: Real_nutty
   Date: 2024-07-21 16:22:45
   Score: 330
   Link: https://reddit.com/r/csMajors/comments/1e8zn36/a_state_your_iq_in_your_resumes/
   Files downloaded (1):
      - 1e8zn36_direct.jpeg (from direct_post)

4. Post ID: 1nnsdyn
   Title: does ts mean I got interview or is it another resume screening for Google
   Author: ChiuceBox
   Date: 2025-09-22 1

## Optional: Search with different queries

You can run additional searches with different keywords to find more resumes:

```python
# Example: Search for additional resume-related terms
additional_queries = ["cv", "roast my resume", "resume review", "critique my resume", "feedback"]

for query in additional_queries:
    print(f"\n{'='*60}")
    print(f"Searching for: '{query}'")
    print(f"{'='*60}")
    
    params = {
        'q': query,
        'restrict_sr': 'on',
        'sort': 'relevance',
        't': 'all',
        'limit': 100
    }
    
    data = fetch_reddit_json(base_url, params)
    
    if data and 'data' in data:
        posts = data['data'].get('children', [])
        
        for post in posts:
            try:
                # Check if we already have this post
                post_id = post['data'].get('id', '')
                if any(p['post_id'] == post_id for p in all_posts_data):
                    continue
                
                post_data, file_count = process_reddit_post(post)
                if file_count > 0:
                    all_posts_data.append(post_data)
                    total_files_downloaded += file_count
                    posts_with_files += 1
                    print(f"  ✓ Found {file_count} file(s) in: {post_data['title'][:50]}...")
            except Exception as e:
                continue
        
        time.sleep(2)  # Be respectful

print(f"\nTotal after additional searches:")
print(f"  Posts: {len(all_posts_data)}")
print(f"  Files: {total_files_downloaded}")
```

## Setup Instructions

### 1. Install Required Packages

Simply install the required Python packages:

```bash
pip install requests pandas tqdm
```

That's it! No Reddit API credentials needed!

### 2. How it works

This scraper uses Reddit's **public JSON API**, which doesn't require authentication. 

- Just add `.json` to any Reddit URL to get JSON data
- Example: `https://reddit.com/r/csMajors/search.json?q=resume`
- No rate limits for reasonable use
- Simpler and faster than the official API

### 3. Run the notebook

Just run all cells from top to bottom!

In [ ]:
# Optional: Load previously saved data
# If you've already run the scraper and want to analyze the data again, run this cell

try:
    with open(output_dir / "metadata.json", 'r') as f:
        all_posts_data = json.load(f)
    
    posts_df = pd.read_csv(output_dir / "posts_summary.csv")
    
    print(f"Loaded {len(all_posts_data)} posts from metadata.json")
    print(f"Total files in dataset: {posts_df['num_files'].sum()}")
except FileNotFoundError:
    print("No saved data found. Run the scraper first.")

In [ ]:
# Create a README file in the output directory
readme_content = f"""# Reddit Resume Dataset - r/csMajors

## Overview
This dataset contains student resumes scraped from the r/csMajors subreddit.

## Scraping Details
- **Subreddit**: r/{SUBREDDIT_NAME}
- **Search Query**: "{SEARCH_QUERY}"
- **Scrape Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- **Total Posts Processed**: {len(all_posts_data)}
- **Posts with Resumes**: {len([p for p in all_posts_data if len(p['downloaded_files']) > 0])}
- **Total Files Downloaded**: {sum(len(p['downloaded_files']) for p in all_posts_data)}

## Dataset Structure

```
resumes/
├── files/              # All downloaded resume files
├── metadata.json       # Complete metadata for all posts and files
├── posts_summary.csv   # Summary table of all posts
└── README.md          # This file
```

## Metadata Fields

### metadata.json
Each post entry contains:
- `post_id`: Reddit post ID
- `title`: Post title
- `author`: Reddit username of poster
- `created_utc`: Unix timestamp of post creation
- `created_date`: Human-readable date
- `url`: Direct URL from the post
- `selftext`: Text content of the post
- `score`: Reddit score (upvotes - downvotes)
- `num_comments`: Number of comments
- `permalink`: Link to the Reddit post
- `downloaded_files`: Array of file information
  - `filename`: Name of the downloaded file
  - `source`: Where the file came from (direct_post, reddit_media, imgur, reddit_gallery)
  - `url`: Original URL of the file

### posts_summary.csv
Simplified table with key information for quick analysis.

## File Naming Convention

Files are named using the pattern: `{{post_id}}_{{source}}_{{index}}.{{extension}}`

Examples:
- `abc123_direct.jpg` - Direct image from post
- `xyz789_imgur_0.png` - First imgur image from post
- `def456_gallery_2.jpg` - Third image from Reddit gallery

## Usage Notes

- All files are downloaded as-is from Reddit and Imgur
- Some files may be images of resumes rather than PDF documents
- File quality and format vary based on what users uploaded
- Metadata preserves attribution to original posters

## Ethics and Privacy

- All data is publicly posted on Reddit
- Usernames are preserved as they appear publicly
- If you use this data, please respect users' privacy
- Consider reaching out to users before using their resumes for any commercial purpose

## Citation

If you use this dataset, please cite:
- Source: r/csMajors subreddit (https://reddit.com/r/csMajors)
- Collection Date: {datetime.now().strftime('%Y-%m-%d')}
"""

readme_path = output_dir / "README.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"README created at: {readme_path.absolute()}")